## Imports

In [26]:
import copy
import salvus.namespace as sn
import numpy as np
import matplotlib.pyplot as plt

import sys
import importlib
from pathlib import Path
path_to_add = str(Path.cwd() / "tomography")
if path_to_add not in sys.path:
    sys.path.append(path_to_add)

import my_code.utilities
importlib.reload(my_code.utilities)
from my_code.utilities import *
import salvus

import salvus.mesh.layered_meshing as lm
from datetime import datetime
import salvus.flow.simple_config as sc
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)

import xarray as xr

## Configuration, project and mesh

In [27]:
# ---- Salvus site --------------------------------------------------------
SITE_NAME = "isambard_oliver"
RANKS = 8

# ---- Physical parameters ------------------------------------------------
CENTRAL_FREQUENCY = 2e4   # Hz
f_c               = 2e4   # centre frequency [Hz]
VP                = 5000.0
RHO               = 2600.0

# ---- Domain -------------------------------------------------------------
x0, x1 = 0.0, 1.0
y0, y1 = 0.0, 1.0

# ---- Project ------------------------------------------------------------
PROJECT_DIR_WIN = '/home/b6as/oliverwfy.b6as/workspace/acoustic_model/Project'
DATA_DIR_WIN    = '/home/b6as/oliverwfy.b6as/workspace/acoustic_model/data'
IMAGE_DIR_WIN   = '/home/b6as/oliverwfy.b6as/workspace/acoustic_model/image'

Path(PROJECT_DIR_WIN).mkdir(parents=True, exist_ok=True)
Path(DATA_DIR_WIN).mkdir(parents=True, exist_ok=True)
Path(IMAGE_DIR_WIN).mkdir(parents=True, exist_ok=True)

PROJECT_NAME = 'acoustic_forward_5paris_salvus'
domain = sn.domain.dim2.BoxDomain(x0=x0, x1=x1, y0=y0, y1=y1)

p = sn.Project.from_domain(
    path=Path(PROJECT_DIR_WIN, PROJECT_NAME), domain=domain, load_if_exists=True
)

# ---- Mesh resolution ----------------------------------------------------
elements_per_wavelength = 3
model_order             = 4
reference_velocity      = 5000
number_of_wavelengths   = 2
reference_frequency     = CENTRAL_FREQUENCY * 2
free_surfaces           = ['y0', 'y1']



# ---- Homogeneous starting model -----------------------------------------
m_homo = sn.material.from_params(rho=RHO, vp=VP)

ab_params = salvus.mesh.simple_mesh.basic_mesh.AbsorbingBoundaryParameters(
    free_surface=free_surfaces,
    number_of_wavelengths=number_of_wavelengths,
    reference_velocity=reference_velocity,
    reference_frequency=reference_frequency,
)

mesh_res = sn.MeshResolution(
    reference_frequency=reference_frequency,
    elements_per_wavelength=elements_per_wavelength,
    model_order=model_order,
)

mesh_homo = lm.mesh_from_domain(
    domain=domain,
    model=sn.layered_meshing.MeshingProtocol(m_homo, ab=ab_params),
    mesh_resolution=mesh_res,
)


# ---- True (layered) model -----------------------------------------------

# mesh_true = sn.layered_meshing.LayeredModel([
#     sn.material.from_params(rho=RHO, vp=VP),
#     sn.layered_meshing.interface.Hyperplane.at(0.6),
#     sn.material.from_params(rho=RHO, vp=1.5 * VP),
#     sn.layered_meshing.interface.Hyperplane.at(0.4),
#     sn.material.from_params(rho=RHO, vp=VP),
# ])

# mesh = lm.mesh_from_domain(
#     domain=domain,
#     model=sn.layered_meshing.MeshingProtocol(mesh_true, ab=ab_params),
#     mesh_resolution=mesh_res,
# )

mesh_true = mesh_homo.copy()
centroids = mesh_true.get_element_centroid()
defect_mask = (centroids[:,1] <= 0.6) & (centroids[:,1] >= 0.4)

mesh_true.elemental_fields["VP"][defect_mask] = 1.5*VP

# ---- region of interest (ROI) -----------------------------------------------

def generate_mesh_roi(mesh, bounds):
    """
    bounds: list of (min, max) per spatial axis, e.g. [(x0, x1), (y0, y1)]
    """
    centroids = mesh.get_element_centroid()
    roi = np.ones(len(centroids), dtype=bool)
    for axis, (lo, hi) in enumerate(bounds):
        roi &= (centroids[:, axis] >= lo) & (centroids[:, axis] <= hi)

    mesh_roi = mesh.copy()
    for field in list(mesh_roi.elemental_fields):
        mesh_roi.elemental_fields.pop(field)
    mesh_roi.attach_field(
        "region_of_interest",
        np.broadcast_to(roi[:, None], mesh_roi.connectivity.shape),
    )
    return mesh_roi


roi = [(0.0, 1.0), (0.4, 0.6)]
mesh_roi = generate_mesh_roi(mesh_homo, roi)
mesh_roi

Accordion(children=(HTML(value='\n                <head>\n                <style>\n                td {\n     …

[2026-04-02 14:59:52,351] INFO: Loading project from /home/b6as/oliverwfy.b6as/workspace/acoustic_model/Project/acoustic_forward_5paris_salvus.


In [14]:
mesh_layered.element_nodal_fields["VP"][defect_mask]

array([[7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [7500.],
       [

In [28]:
p.viz.nb.inversion(inverse_problem_configuration="inversion_L2")